---
title: "Ionosphere modelling with kriging"
bibliography: ../bibliography.bib
---

# Ionosphere modelling with kriging

In this guide, we will show how to use the kriging tools built into GNX-py to build a regional VTEC model from measured GNSS STEC. Kriging is a geostatistical interpolation method: it estimates a quantity at locations where we do not have direct measurements by combining neighbouring observations with weights derived from a spatial covariance/variogram model. In ionospheric modelling, the measured quantity is usually VTEC at ionospheric pierce points (IPPs), and the target is a regular grid of ionospheric grid points (IGPs).

The intuition is simple. If two IPPs are close to each other on the ionospheric shell, we expect their VTEC values to be more similar than values observed hundreds or thousands of kilometres apart. Kriging makes this intuition explicit through the variogram:

$$
\gamma(h) = \frac{1}{2}\operatorname{Var}\left[Z(\mathbf{x}) - Z(\mathbf{x}+\mathbf{h})\right]
$$

where $Z(\mathbf{x})$ is the VTEC field and $h = \|\mathbf{h}\|$ is the spatial separation. Equivalently, we can work with covariance:

$$
C(h) = \operatorname{Cov}\left[Z(\mathbf{x}), Z(\mathbf{x}+\mathbf{h})\right]
$$

For a target point $\mathbf{x}_0$, the kriging estimate is a weighted sum of nearby measurements:

$$
\widehat{Z}(\mathbf{x}_0) = \sum_{i=1}^{N} w_i Z(\mathbf{x}_i)
$$

The weights are not chosen only by inverse distance. They are solved from a linear system that uses the covariance/variogram between all selected IPPs and between each IPP and the target point. This is why kriging can produce both an estimate and a formal interpolation variance. The variance should not be interpreted as a full physical error budget, but it is a useful diagnostic: areas far from observations, or constrained by weak geometry, should have larger uncertainty.

GNX-py currently exposes three kriging modes through `IonoKrigingMonitor`:

| Mode | Idea | Practical interpretation |
|---|---|---|
| `OK` | Ordinary Kriging | Assumes an unknown but locally constant mean. Good as a classical baseline. |
| `UK` | Universal Kriging | Adds a regional linear drift term, useful when the VTEC field has a large-scale slope across the region. |
| `WAAS` | Sparks/WAAS-style local kriging | Uses a local covariance model and neighbourhood selection around each grid point; this is the most ionosphere-oriented workflow in the current tutorial API. |

In this notebook we use the same measured STEC network as in the activity-index tutorial. The satellite DCBs and receiver DCBs have already been handled during the STEC preparation workflow, so we can convert levelled STEC to VTEC and interpolate it. This is crucial: unlike SIDX or ROTI, kriging compares different receiver-satellite links at the same epoch, so absolute levelling and bias consistency matter.

Kriging offers two major advantages in regional ionosphere modelling. First, it can generate a regional VTEC map at a finer grid spacing than products such as GIM. Second, it uses real measurements from the analysed epoch, so local structures can be visible more clearly than in empirical models such as Klobuchar, NeQuick, or NTCM. The cost is that kriging is not a time-propagation model: each map is estimated from measurements available at that epoch. It also depends strongly on the quality of calibration, station geometry, variogram choice, and neighbourhood selection.

The goal here is not to prove that one model is universally better. Instead, we will learn the GNX-py workflow: load measured STEC, prepare VTEC, choose a grid, configure `IonoKrigingMonitor`, generate a small set of maps, inspect the output dataset, and compare the regional model against GIM and NTCM at common grid points.

In [ ]:
from pathlib import Path
import importlib.util
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

import gnx_py as gnx

warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in create_collection",
    category=RuntimeWarning,
    module="shapely.creation",
)
warnings.filterwarnings(
    "ignore",
    message="The PostScript backend does not support transparency; partially transparent artists will be rendered opaque",
)


def show_or_close(fig=None):
    """Display figures in notebooks and render/close during headless smoke runs."""
    fig = fig or plt.gcf()
    if "agg" in plt.get_backend().lower():
        fig.canvas.draw()
        plt.close(fig)
    else:
        plt.show()


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "gnx_py").is_dir() and (candidate / "tutorial").is_dir():
            return candidate
    raise RuntimeError("Could not find the GNX repository root. Start Jupyter from inside the cloned repository.")


def load_local_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


REPO_ROOT = find_repo_root()
TUTORIAL_DIR = REPO_ROOT / "tutorial" / "ionosphere"
NETWORK_DIR = TUTORIAL_DIR / "network"
OUTPUT_DIR = TUTORIAL_DIR / "output" / "kriging"
FIGURE_DIR = TUTORIAL_DIR / "figures" / "kriging"
SAMPLE_DATA_DIR = REPO_ROOT / "sample_data"

vv = load_local_module("kriging_vtec_viz", TUTORIAL_DIR / "vtec_viz.py")

print(f"Repository root: {REPO_ROOT}")
print(f"Network folder:  {NETWORK_DIR}")
print(f"GNX-py module:   {gnx.__file__}")

First, let us define a few parameters that control the code. `PLOT_EVERY_MIN` is the time interval at which we want to generate VTEC snapshots. For this tutorial we keep it at 60 minutes, so the run produces hourly maps. The analysis window is short on purpose: the supplied `network/` folder covers the full day, but kriging should be learned on a small run before scaling up.

`IONO_SHELL_HEIGHT_M` is the height used by the thin-shell mapping from STEC to VTEC. `ZENITH_SIGMA_TECU` is a simple measurement-noise proxy at zenith. We convert it to an elevation-dependent variance by scaling with $1/\sin(e)$:

$$
\sigma_i(e)^2 = \left(\frac{\sigma_0}{\sin e_i}\right)^2
$$

This is not a complete stochastic model of STEC calibration errors. It is a compact tutorial choice that tells the WAAS/Sparks solver that low-elevation observations should be trusted less.

In [ ]:
PLOT_EVERY_MIN = 60
ANALYSIS_START = pd.Timestamp("2024-02-04 12:00:00", tz="UTC")
ANALYSIS_END = pd.Timestamp("2024-02-04 14:00:00", tz="UTC")

IONO_SHELL_HEIGHT_M = 450e3
ZENITH_SIGMA_TECU = 1.5
MIN_ELEV_DEG = 30


def is_tick(ts: pd.Timestamp) -> bool:
    ts = pd.Timestamp(ts)
    if PLOT_EVERY_MIN > 60:
        plot_every_h = PLOT_EVERY_MIN // 60
        return (ts.second == 0) and (ts.hour % plot_every_h == 0) and (ts.minute == 0)
    return (ts.second == 0) and (ts.minute % PLOT_EVERY_MIN == 0)

Next, we set the paths where the network measurements, generated models, and tutorial figures are located. The important part is that all paths are derived from the repository root. The notebook can therefore be executed from Jupyter, an IDE, or a converted notebook runner without assuming the current working directory is `tutorial/ionosphere`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

GIM_PATH = SAMPLE_DATA_DIR / "COD0OPSFIN_20240350000_01D_01H_GIM.INX"
NAV_PATH = SAMPLE_DATA_DIR / "BRDC00IGS_R_20240350000_01D_MN.rnx"

required_paths = [NETWORK_DIR, GIM_PATH, NAV_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing tutorial input data:\n" + "\n".join(missing))

print(f"Network STEC files: {len(list(NETWORK_DIR.glob('*STEC.parquet.gzip')))}")
print(f"Output directory:   {OUTPUT_DIR}")
print(f"Figure directory:   {FIGURE_DIR}")

Data loading and preparation is performed in the same spirit as for ionospheric activity indices. The difference is interpretation: for SIDX and ROTI we mostly cared about temporal changes along a link, whereas for kriging we need a consistent VTEC value at each IPP.

The loader returns levelled STEC in TECU. We then:

1. reset the index so that `sv` and `time` become columns,
2. compute the mapping function $M(e;H)$,
3. create `vtec = stec / M`,
4. add receiver and satellite identifiers for optional covariance/bias terms,
5. add `meas_var`, the elevation-dependent measurement variance used by the WAAS/Sparks mode,
6. keep a short analysis window and a slightly extended spatial region around the target grid.

In [ ]:
df_all, load_info = gnx.load_stec_folder(
    folder=NETWORK_DIR,
    min_elev_deg=MIN_ELEV_DEG,
    quiet=True,
    skip_negative=True,
)

df_all = df_all.reset_index().copy()
df_all["time"] = pd.to_datetime(df_all["time"], utc=True)
df_all["sv"] = df_all["sv"].astype(str).str[:3]
df_all["stec"] = df_all["leveled_tec"]
df_all["mf"] = gnx.mapping_M(df_all["ev"].to_numpy(), H_m=IONO_SHELL_HEIGHT_M)
df_all["vtec"] = df_all["stec"] / df_all["mf"]
df_all["rec_id"] = df_all["name"]
df_all["sat_id"] = df_all["sv"]

sigma_elev = ZENITH_SIGMA_TECU / np.sin(np.deg2rad(df_all["ev"].to_numpy()))
df_all["meas_var"] = sigma_elev**2

spatial_mask = df_all["lat_ipp"].between(45.0, 60.0) & df_all["lon_ipp"].between(5.0, 35.0)
time_mask = (df_all["time"] >= ANALYSIS_START) & (df_all["time"] < ANALYSIS_END)
df_work = df_all.loc[spatial_mask & time_mask].sort_values("time").copy()

print({k: load_info[k] for k in ["files_scanned", "files_accepted", "files_rejected"]})
print(f"Prepared rows: {len(df_work):,}")
print(f"Epochs in tutorial window: {df_work['time'].nunique()}")

Next, we define the grid on which we interpolate VTEC. The original version of this tutorial used a very dense $0.1^\circ \times 0.1^\circ$ grid. That is useful for final figures, but it is unnecessarily heavy while learning the workflow. Here we use $0.5^\circ$ spacing over the same regional area.

The important conceptual distinction is this:

- IPPs are the irregular measurement locations,
- IGPs are the regular output grid points,
- kriging estimates VTEC at IGPs from nearby IPPs.

You can reduce `GRID_STEP_DEG` later when you want publication-quality maps.

In [ ]:
lat_min, lat_max = 50.0, 57.5
lon_min, lon_max = 10.0, 30.0
GRID_STEP_DEG = 0.5

lat_grid = np.arange(lat_min, lat_max + 0.5 * GRID_STEP_DEG, GRID_STEP_DEG)
lon_grid = np.arange(lon_min, lon_max + 0.5 * GRID_STEP_DEG, GRID_STEP_DEG)
region = (lon_min, lon_max, lat_min, lat_max)

print(f"Grid shape: {lat_grid.size} x {lon_grid.size} = {lat_grid.size * lon_grid.size:,} points")

The heart of the kriging module in GNX-py is the `IonoKrigingMonitor` class. It separates the stable configuration of a grid and method from the epoch-by-epoch input measurements. This is useful because in an operational workflow we usually keep the same grid and stochastic assumptions, while the IPPs change at every epoch.

For the current tutorial we use `kriging_mode="WAAS"`, based on the Sparks/WAAS-style local approach. The core idea is local: for each IGP, the algorithm selects neighbouring IPPs and solves a covariance-based interpolation problem. The relevant parameters are:

- `Rmin_km`, `Rmax_km` — inner and outer search radii around each grid point,
- `Ntarget` — preferred number of IPPs used in the local solution,
- `Nmin` — minimum number of IPPs required to produce an estimate,
- `variogram` — covariance model that controls how quickly VTEC correlation decays with distance,
- `default_meas_var` and `meas_var` — measurement-noise variance if it is not provided or is provided per observation.

The Sparks variogram used here can be read as:

$$
C(d) = (\sigma_{\mathrm{total}}^2 - \sigma_{\mathrm{nominal}}^2)\exp\left(-\frac{d}{d_{\mathrm{decorr}}}\right), \qquad d > 0
$$

and

$$
C(0) = \sigma_{\mathrm{total}}^2
$$

where $d_{\mathrm{decorr}}$ controls how quickly measurements become decorrelated with distance. Larger values spread information farther away; smaller values make the map more local and usually increase uncertainty away from dense IPP coverage.

`OK` and `UK` are also available. In GNX-py they use classical semivariogram parameters `sill`, `range`, and `nugget`; `UK` additionally includes a regional linear drift. They are useful for experiments and comparisons, but the WAAS/Sparks mode is the most directly tailored to this ionosphere workflow.

In [ ]:
variogram = gnx.SparksVariogram(
    sigma_nominal=0.5,
    sigma_total=1.5,
    ddecorr_km=800.0,
)

mon = gnx.IonoKrigingMonitor(
    grid_lon=lon_grid,
    grid_lat=lat_grid,
    coord_frame="GEO",
    sm_transformer=None,
    sm_obstime=None,
    kriging_mode="WAAS",
    Rmin_km=600.0,
    Rmax_km=1800.0,
    Ntarget=25,
    Nmin=8,
    variogram=variogram,
    default_meas_var=ZENITH_SIGMA_TECU**2,
)
mon

We prepare containers for storing epoch results and run kriging for the selected grid in a loop. Each epoch is handled independently. In this tutorial, `PLOT_EVERY_MIN=60`, so from a two-hour window we obtain two maps: 12:00 and 13:00 UTC.

The returned arrays are:

- `V` — gridded VTEC estimate in TECU,
- `SS` — formal kriging variance in TECU$^2$,
- `info` — diagnostic metadata such as mode, number of input points and neighbourhood settings.

We save the small result to NetCDF because this is the format used by the rest of the tutorial tooling and by xarray.

In [ ]:
times_all = []
V_all = []
SS_all = []
info_all = []

for epoch_time, df_epoch in df_work.groupby("time"):
    if not is_tick(epoch_time):
        continue
    V, SS, info = mon.krige_epoch(df_epoch, epoch_time=epoch_time)
    times_all.append(epoch_time)
    V_all.append(V.astype(np.float32))
    SS_all.append(SS.astype(np.float32))
    info_all.append(info)

if not times_all:
    raise RuntimeError("No kriging epochs were selected. Check PLOT_EVERY_MIN and the analysis time window.")

ds = gnx.save_grids_to_netcdf(
    times=times_all,
    V_all=V_all,
    SS_All=SS_all,
    lat_grid=lat_grid,
    lon_grid=lon_grid,
    out_path=str(OUTPUT_DIR),
    out_name="model",
)

print(f"Generated epochs: {len(times_all)}")
print(info_all[0])

The results are stored in `xarray.Dataset` format and saved in NetCDF format. The variables stored in the dataset are `V` (VTEC) and `SS` (kriging variance). The data layout is `(time, lat, lon)`, which makes it convenient to select time slices, compute statistics, compare with GIM/NTCM, and plot maps.

To display the model we use `vtec_viz.py`, a tutorial helper that keeps maps consistent for Markdown/PDF conversion.

In [ ]:
print(ds.data_vars)

In [ ]:
list(ds.coords)

In [ ]:
int(len(ds.time))

In [ ]:
model = mon.kriging_mode
t_index = len(ds.time) // 2
time = pd.Timestamp(ds["time"][t_index].values)
hh = time.hour

vtec_map_path = FIGURE_DIR / f"{model}_MAP_{hh:02d}.png"
var_map_path = FIGURE_DIR / f"{model}_VAR_{hh:02d}.png"

vv.plot_model(
    ds=ds,
    var="V",
    t_index=t_index,
    region=region,
    cbar_orientation="horizontal",
    cbar_location="bottom",
    cbar_shrink=0.8,
    cbar_aspect=40,
    cbar_label="VTEC [TECU]",
    cbar_pad=0.0,
    draw_coastlines=True,
    figsize=(10, 10),
    cmap="viridis",
    title=f"VTEC map @ {time}",
    savepath=str(vtec_map_path),
    show=False,
)

vv.plot_model(
    ds=ds,
    var="SS",
    t_index=t_index,
    region=region,
    cbar_orientation="horizontal",
    cbar_location="bottom",
    cbar_shrink=0.8,
    cbar_aspect=40,
    cbar_label=r"Kriging variance [TECU$^2$]",
    cbar_pad=0.0,
    draw_coastlines=True,
    figsize=(10, 10),
    cmap="viridis",
    title=f"Kriging variance map @ {time}",
    savepath=str(var_map_path),
    show=False,
)

display(Image(filename=str(vtec_map_path), width=760))
display(Image(filename=str(var_map_path), width=760))

Let us compare our model with NTCM and GIM. It is worth recalling what we said at the beginning: GIM is a global assimilation product with a coarser native grid, while our kriging grid is regional and measurement-driven. Therefore this comparison should not be read as a definitive ranking of models. It is a diagnostic exercise:

- do the regional VTEC levels look plausible against GIM?
- how far is a simple empirical model such as NTCM from the measurement-driven map?
- are differences spatially coherent or dominated by isolated artefacts?

First, we load the IONEX file using GNX-py.

In [ ]:
reader = gnx.GIMReader(tec_path=GIM_PATH)
ionex = reader.read()
inx_tec = ionex.tec
inx_tec

To calculate TEC from the NTCM model, we need Galileo ionospheric model coefficients from the navigation message. This gives us a useful empirical-model baseline on the same grid and at the same epochs as the kriging model.

In [ ]:
processor = gnx.GNSSDataProcessor2(nav_path=NAV_PATH)
hdr = processor.nav_header_reader()
gal_alpha = hdr[-1]
gal_alpha

We now generate an NTCM grid corresponding to the kriging grid. The arguments are the latitude vector, longitude vector, epoch list, and Galileo ionospheric coefficients.

In [ ]:
lat_grid_cmp = ds["V"].lat.values
lon_grid_cmp = ds["V"].lon.values
times_cmp = ds.time.values

ds_ntcm = gnx.compute_ntcm_grid(lat_grid_cmp, lon_grid_cmp, times_cmp, gal_alpha=gal_alpha)
ds_ntcm

Datasets are convenient because comparison can be performed at common coordinates. The helper `compare_plot` displays the absolute TEC difference between two datasets and returns mean and median absolute differences.

Keep in mind that when comparing kriging and GIM, the goal is not to show which model is objectively “better”. They are produced using different assumptions, resolutions and input data. Here we use GIM as a reference surface for a quick sanity check of regional VTEC levels.

In [ ]:
mns = []
mns_ntcm = []
comparison_time_index = len(ds.time) // 2
kriging_compare_path = FIGURE_DIR / "kriging_minus_gim.png"
ntcm_compare_path = FIGURE_DIR / "ntcm_minus_gim.png"

for num, epoch_time in enumerate(ds.time.values):
    show_this = num == comparison_time_index

    fig, ax, mean_abs, median_abs = vv.compare_plot(
        ds_model=ds,
        ds_ref=inx_tec,
        time=epoch_time,
        var_model="V",
        var_ref="TEC",
        region=region,
        cmap="Blues",
        cbar_orientation="horizontal",
        cbar_pad=0.1,
        cbar_shrink=0.8,
        cbar_aspect=40,
        figsize=(5, 5),
        title_prefix=r"|ΔTEC| = |TEC$_\mathrm{GIM}$ - TEC$_\mathrm{KRIGING}$|",
        show=show_this,
    )
    mns.append(mean_abs)
    if show_this:
        fig.savefig(kriging_compare_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

    fig, ax, mean_abs, median_abs = vv.compare_plot(
        ds_model=ds_ntcm,
        ds_ref=inx_tec,
        time=epoch_time,
        var_model="V",
        var_ref="TEC",
        region=region,
        cmap="Blues",
        cbar_orientation="horizontal",
        cbar_pad=0.1,
        cbar_shrink=0.8,
        cbar_aspect=40,
        figsize=(5, 5),
        title_prefix=r"|ΔTEC| = |TEC$_\mathrm{GIM}$ - TEC$_\mathrm{NTCM}$|",
        show=show_this,
    )
    mns_ntcm.append(mean_abs)
    if show_this:
        fig.savefig(ntcm_compare_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

display(Image(filename=str(kriging_compare_path), width=520))
display(Image(filename=str(ntcm_compare_path), width=520))

As you can see, the TEC differences between NTCM and GIM are not the same type of signal as the differences between kriging and GIM. NTCM is an empirical broadcast-style model; kriging is a measurement-driven regional interpolation. Finally, let's plot the mean absolute error at each generated epoch on a common graph.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(
    "Mean absolute VTEC difference against GIM\n"
    f"Kriging: {np.mean(np.asarray(mns)):.3f} TECU | "
    f"NTCM: {np.mean(np.asarray(mns_ntcm)):.3f} TECU"
)
ax.plot(range(len(mns)), mns, marker="o", label="MAE for Kriging")
ax.plot(range(len(mns_ntcm)), mns_ntcm, marker="o", label="MAE for NTCM")
ax.grid(color="k", linestyle="-", linewidth=0.5, alpha=0.35)
ax.set_xlabel("Map number")
ax.set_ylabel("MAE [TECU]")
ax.legend()
show_or_close(fig)

You now know the practical GNX-py kriging workflow. Together with the functions that calculate ionospheric activity indices, kriging can be used to monitor and visualise regional electron content changes during quiet and disturbed ionospheric conditions.

A few practical rules are worth remembering:

- kriging needs calibrated or consistently levelled STEC/VTEC,
- the result quality depends on station geometry and IPP coverage,
- variogram parameters are not universal constants and should be validated for the region and time period,
- `SS` is a formal interpolation variance, not a complete physical error estimate,
- `OK`, `UK`, and `WAAS` answer slightly different modelling questions, so compare them before adopting one as default.

You already know the SPP and PPP modules. A useful follow-up experiment is to connect a kriging-derived VTEC model to positioning and compare positioning residuals against GIM, Klobuchar/NTCM, or unmodelled ionosphere cases.

The VTEC model generated using kriging is stored in xarray/NetCDF format. The file can be loaded with `xarray.load_dataset()` or `xarray.open_dataset()`:

```python
import xarray as xr
loaded = xr.load_dataset(TUTORIAL_DIR / "output" / "kriging" / "model.nc")
```

The xarray library is convenient for grid data because it keeps dimensions, coordinates and variables together. That makes it natural to select epochs, interpolate, compare with GIM-like products, and export figures for reports.